# Building a Healthcare Analytics Dataset from Raw Multi-Source Data

## Environment setup and synthetic data

**Important — Do not modify these datasets**

The synthetic datasets below are fixed and used by all learners.
Do not modify this cell or change any data values, as **grading is based on the exact outputs** produced from these datasets.

You should only write code in the sections of the notebook that are clearly marked for you to complete.

In [2]:
# ⚠️ DO NOT MODIFY THIS CELL
# This dataset is fixed and used for consistent grading across all learners.

import pandas as pd
import numpy as np

ehr_df = pd.DataFrame({
    "patient_id": ["P001","P002","P003","P003","P004"],
    "patient_name": ["John Doe","Jane Smith","Alice Brown","Alice Brown","Bob Lee"],
    "dob": pd.to_datetime(["1980-05-12","1975-09-30","1990-02-15","1990-02-15","1965-11-01"]),
    "encounter_id": ["E1","E2","E3","E3","E4"],
    "encounter_date": pd.to_datetime(["2023-01-10","2023-01-12","2023-01-15","2023-01-15","2023-01-18"]),
    "heart_rate": [72,-10,85,85,220]
})

claims_df = pd.DataFrame({
    "member_id": ["001","002","003","004"],
    "diagnosis_code": ["I10","E11","E11","I10"],
    "claim_amount": [12000,18000,15000,20000]
})

labs_df = pd.DataFrame({
    "lab_patient_id": ["P-001", "P-002", "P-003", "P-004"],
    "lab_date": pd.to_datetime(["2023-01-08", "2023-01-09", "2023-01-14", "2023-01-17"]),
    "glucose": [95, 5.5, 110, 6.2],
    "unit": ["mg/dL", "mmol/L", "mg/dL", "mmol/L"]
})

### Step 1: Inspect dataset structure

**What is expected**
- Display the number of rows and columns of the EHR dataset
- Display all column names and their data types
(For example, by showing the dataset shape and column data types.)

In [3]:
# your code goes here

print(f"Number of rows: {ehr_df.shape[0]}")
print(f"Number of columns: {ehr_df.shape[1]}")

Number of rows: 5
Number of columns: 6


### Step 2: Identify duplicate records

**What is expected**
- Print the total number of duplicate rows
- Show which rows are duplicates using a clear row-level indicator (For example, by displaying a row-level indicator showing True/False for duplicates.)


In [4]:
# your code goes here
print(ehr_df.duplicated().sum())
ehr_df[ehr_df.duplicated()]


1


,patient_id,patient_name,dob,encounter_id,encounter_date,heart_rate
3,P003,Alice Brown,1990-02-15,E3,2023-01-15,85


### Step 3: Remove duplicate records

**What is expected**
- Remove duplicate rows from the dataset
- Print the updated dataset shape (rows and columns)

In [5]:
# your code goes here
ehr_df_deduped = ehr_df.drop_duplicates().copy()
ehr_df_deduped

,patient_id,patient_name,dob,encounter_id,encounter_date,heart_rate
0,P001,John Doe,1980-05-12,E1,2023-01-10,72
1,P002,Jane Smith,1975-09-30,E2,2023-01-12,-10
2,P003,Alice Brown,1990-02-15,E3,2023-01-15,85
4,P004,Bob Lee,1965-11-01,E4,2023-01-18,220


### Step 4: Identify invalid clinical values

**What is expected**
- Print any rows where heart rate values are invalid (less than zero)

In [6]:
# your code goes here
ehr_df_deduped[ehr_df_deduped["heart_rate"] < 0]

,patient_id,patient_name,dob,encounter_id,encounter_date,heart_rate
1,P002,Jane Smith,1975-09-30,E2,2023-01-12,-10


### Step 5: Correct invalid clinical values

**What is expected**
- Replace invalid heart rate values with missing values (NaN)
- Print the updated heart rate column

In [7]:
# your code goes here

ehr_df_deduped[ehr_df_deduped["heart_rate"] < 0 ] = np.nan
ehr_df_deduped[["heart_rate"]]

,heart_rate
0,72.0
1,NaN
2,85.0
4,220.0


### Step 6: Standardize patient identifiers across sources

**What is expected**
- Create a standardized patient identifier column
- Print the standardized identifiers from all three datasets

In [16]:
# your code goes here
# Standardize EHR Patient ID
ehr_df["patient_id"] = ehr_df["patient_id"].str.replace("P", "")

# Standardize Lab Patient ID
labs_df["patient_id"] = labs_df["lab_patient_id"].str.replace("P-", "")

# Standardize Claims Patient ID
claims_df["patient_id"] = claims_df["member_id"].astype(str).str.zfill(3)

ehr_df, labs_df, claims_df

(  patient_id patient_name        dob encounter_id encounter_date  heart_rate
 0        001     John Doe 1980-05-12           E1     2023-01-10          72
 1        002   Jane Smith 1975-09-30           E2     2023-01-12         -10
 2        003  Alice Brown 1990-02-15           E3     2023-01-15          85
 3        003  Alice Brown 1990-02-15           E3     2023-01-15          85
 4        004      Bob Lee 1965-11-01           E4     2023-01-18         220,
   lab_patient_id   lab_date  glucose    unit patient_id
 0          P-001 2023-01-08     95.0   mg/dL        001
 1          P-002 2023-01-09      5.5  mmol/L        002
 2          P-003 2023-01-14    110.0   mg/dL        003
 3          P-004 2023-01-17      6.2  mmol/L        004,
   member_id diagnosis_code  claim_amount patient_id
 0       001            I10         12000        001
 1       002            E11         18000        002
 2       003            E11         15000        003
 3       004            I10      

### Step 7: Map diagnosis codes to clinical conditions

**What is expected**
- Map diagnosis codes to readable clinical condition names
- Print diagnosis codes alongside mapped conditions

In [19]:
# your code goes here

diagnosis_map = {
    "I10" : "Hypertension",
    "E11" : "Type 2 Diabetes"
}

claims_df["diagnosis_desc"] = claims_df["diagnosis_code"].map(diagnosis_map)

claims_df[["diagnosis_code", "diagnosis_desc"]]

,diagnosis_code,diagnosis_desc
0,I10,Hypertension
1,E11,Type 2 Diabetes
2,E11,Type 2 Diabetes
3,I10,Hypertension


### Step 8: Resolve laboratory unit mismatches

**What is expected**
- Convert all glucose values to a consistent unit (mg/dL)
- Print original values, units, and converted values

In [25]:
# your code goes here

def convert_glucose_to_mgdl(value, unit):
    if unit == "mmol/L":
        return value * 18
    return value

labs_df["glucose_new"] = labs_df.apply(
    lambda row: convert_glucose_to_mgdl(row["glucose"], row["unit"]), axis=1
)

labs_df[["glucose", "glucose_new", "unit"]]

,glucose,glucose_new,unit
0,95.0,95.0,mg/dL
1,1782.0,32076.0,mmol/L
2,110.0,110.0,mg/dL
3,2008.8,36158.4,mmol/L


### Step 9: Merge EHR, claims, and laboratory data

**What is expected**
- Merge all datasets using the standardized patient identifier
- Display the column names to confirm the merge
- Display key fields from each source system (for example, a patient identifier, an encounter field, a diagnosis field, and a laboratory value) to verify the merged result

In [26]:
# your code goes here

merged_df = ehr_df.merge(claims_df, on="patient_id", how="inner").merge(labs_df, on="patient_id", how="inner")

merged_df

,patient_id,patient_name,dob,encounter_id,encounter_date,heart_rate,member_id,diagnosis_code,claim_amount,diagnosis_desc,lab_patient_id,lab_date,glucose,unit,glucose_new
0,001,John Doe,1980-05-12,E1,2023-01-10,72,001,I10,12000,Hypertension,P-001,2023-01-08,95.0,mg/dL,95.0
1,002,Jane Smith,1975-09-30,E2,2023-01-12,-10,002,E11,18000,Type 2 Diabetes,P-002,2023-01-09,1782.0,mmol/L,32076.0
2,003,Alice Brown,1990-02-15,E3,2023-01-15,85,003,E11,15000,Type 2 Diabetes,P-003,2023-01-14,110.0,mg/dL,110.0
3,003,Alice Brown,1990-02-15,E3,2023-01-15,85,003,E11,15000,Type 2 Diabetes,P-003,2023-01-14,110.0,mg/dL,110.0
4,004,Bob Lee,1965-11-01,E4,2023-01-18,220,004,I10,20000,Hypertension,P-004,2023-01-17,2008.8,mmol/L,36158.4


### Step 10: Apply HIPAA safe harbor de-identification

**What is expected**
- Remove the patient name column, which represents a direct identifier in this dataset
- Print the column list to confirm that the patient_name column has been removed
- Create a generalized year-of-birth field from the date of birth (for example, extracting the year)
- To keep the output readable for grading, **print only the patient identifier and birth year columns**

Note: The full dataset should still be updated. You are only limiting what is printed.

In [36]:
# your code goes here

#drop the name
deidentify_df = merged_df.drop(columns=["patient_name"])

#Generalize the dates to year only
deidentify_df["birth_year"] = deidentify_df["dob"].dt.year
deidentify_df["visit_year"] = deidentify_df["encounter_date"].dt.year


#Drop the Original data columns

deidentify_df = deidentify_df.drop(columns=["dob", "encounter_date"])

deidentify_df

,patient_id,encounter_id,heart_rate,member_id,diagnosis_code,claim_amount,diagnosis_desc,lab_patient_id,lab_date,glucose,unit,glucose_new,birth_year,visit_year
0,001,E1,72,001,I10,12000,Hypertension,P-001,2023-01-08,95.0,mg/dL,95.0,1980,2023
1,002,E2,-10,002,E11,18000,Type 2 Diabetes,P-002,2023-01-09,1782.0,mmol/L,32076.0,1975,2023
2,003,E3,85,003,E11,15000,Type 2 Diabetes,P-003,2023-01-14,110.0,mg/dL,110.0,1990,2023
3,003,E3,85,003,E11,15000,Type 2 Diabetes,P-003,2023-01-14,110.0,mg/dL,110.0,1990,2023
4,004,E4,220,004,I10,20000,Hypertension,P-004,2023-01-17,2008.8,mmol/L,36158.4,1965,2023


### Step 11: Create encounter utilization features

**What is expected**
- Count the number of encounters per patient
- Print the encounter count for each patient using the patient identifier

In [38]:
# your code goes here

encounter_counts = deidentify_df.groupby("patient_id")["encounter_id"].count().reset_index(name="encounter_count")

encounter_counts

,patient_id,encounter_count
0,001,1
1,002,1
2,003,2
3,004,1


### Step 12: Engineer final model-ready features

**What is expected**
- Aggregate patient-level features
- Create a final feature table with one row per patient that includes a glucose-related feature and an encounter count
- Print the final feature table

In [39]:
# Aggregate patient-level features
glucose_avg = deidentify_df.groupby("patient_id")["glucose_new"].mean().reset_index(name="avg_glucose")

# Create a final feature table with one row per patient that includes a glucose-related feature and an encounter count
final_features = encounter_counts.merge(glucose_avg, on="patient_id", how="left")

# Print the final feature table
final_features

,patient_id,encounter_count,avg_glucose
0,001,1,95.0
1,002,1,32076.0
2,003,2,110.0
3,004,1,36158.4
